In [1]:
import pandas as pd
import os
from random import randint
import random 
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('/Users/danielgarcia-barnett/Desktop/prog_choice_reaction_time/Stimuli/Schedules/1_option/schedule-001.par', index_col=0, keep_default_na=False)

In [4]:
df

""
0.0000 1 2.000 1.0000 line
2.0000 0 1.000 1.0000 NULL
3.0000 1 2.000 1.0000 line
5.0000 0 1.000 1.0000 NULL
6.0000 1 2.000 1.0000 line
8.0000 0 1.100 1.0000 NULL
...
122.5999 0 1.300 1.0000 NULL
123.8999 1 2.000 1.0000 line
125.8999 0 1.000 1.0000 NULL


In [35]:
def get_sched(fpath):
    with open(fpath, 'r') as f:
        lines = f.readlines()
        lines = [line.split() for line in lines]
    schedule = pd.DataFrame(lines, columns=['time', 'stim_code', 'duration', '?', 'stim_type'])
    schedule = schedule.apply(pd.to_numeric, errors='ignore')
    return schedule
def save_sched(df, fpath):
    df.to_csv(fpath, sep=' ', header=False, index=False)

In [40]:
fpath = '/Users/danielgarcia-barnett/Desktop/prog_choice_reaction_time/Stimuli/Schedules/4_options/schedule-001.par'
get_sched(fpath)

/var/folders/w0/4d261sg51qjdphncfst0x4fr0000gn/T/ipykernel_71347/1754735922.py:6: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  schedule = schedule.apply(pd.to_numeric, errors='ignore')


,time,stim_code,duration,?,stim_type
0,0.0,0,1.0,1.0,NULL
1,1.0,2,2.0,1.0,square
2,3.0,0,1.0,1.0,NULL
3,4.0,3,2.0,1.0,circle
4,6.0,0,1.9,1.0,NULL
...,...,...,...,...,...
76,123.1,0,1.2,1.0,NULL
77,124.3,1,2.0,1.0,line
78,126.3,0,1.1,1.0,NULL
79,127.4,2,2.0,1.0,square


In [39]:
for dirs, fldrs, files in os.walk('/Users/danielgarcia-barnett/Desktop/prog_choice_reaction_time/Stimuli/Schedules'):
    for file in files:
        if file.endswith('.par'):
            fpath = os.path.join(dirs, file)
            df = get_sched(fpath)
            df = pd.concat([df.iloc[41:42], df]).reset_index(drop=True)
            time = [0]+[float(df.loc[0:i, 'duration'].sum()) for i in range(len(df))]
            df['time'] = time[:-1]
            save_sched(df, fpath)

/var/folders/w0/4d261sg51qjdphncfst0x4fr0000gn/T/ipykernel_71347/1754735922.py:6: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  schedule = schedule.apply(pd.to_numeric, errors='ignore')


In [6]:
dirs = '/Users/danielgarcia-barnett/Desktop/prog_choice_reaction_time/Stimuli/Schedules/practice'
stim_list = ['line', 'square', 'circle', 'triangle']*2
for i in range(100):
    random.shuffle(stim_list)
    duration = [np.round(random.uniform(1, 2), 1) for i in range(8)]
    null_data = pd.DataFrame({'duration':duration, 'stim_type':['NULL']*8}).reset_index(drop=True)
    stim_data = pd.DataFrame({'duration':[2]*8, 'stim_type':stim_list}).reset_index(drop=True)
    practice_data = pd.concat([stim_data, null_data], keys=[0,1]).sort_index(level=1).reset_index(drop=True)
    practice_data.to_csv(os.path.join(dirs, f'schedule-{i:03}.csv'))

In [11]:
practice_data

,duration,stim_type
0,2.0,square
1,1.8,NULL
2,2.0,triangle
3,1.5,NULL
4,2.0,circle
5,1.8,NULL
6,2.0,circle
7,1.7,NULL
8,2.0,circle
9,1.4,NULL


In [80]:
df[::-1]

,time,stim_code,duration,?,stim_type
239,419.0,0,1.0,1.0,NULL
238,417.0,1,2.0,1.0,line
237,416.5,0,0.5,1.0,NULL
236,414.5,2,2.0,1.0,square
235,412.5,0,2.0,1.0,NULL
...,...,...,...,...,...
4,5.0,4,2.0,1.0,circle
3,4.5,0,0.5,1.0,NULL
2,2.5,1,2.0,1.0,line
1,2.0,0,0.5,1.0,NULL


# Generating the Order Text Files

In [4]:
order_dirs = '/Users/AP-CNL/Desktop/4RT/Stimuli/Order'

In [8]:
opt_dict = {'1_option':['line'],
            '2_options':['line', 'square'],
            '4_options':['line', 'circle', 'triangle', 'square']}

In [40]:
def get_order(opt_list):
    count = 40 // len(opt_list)
    count_list = [count for i in range(len(opt_list))]
    order_list = []
    i = 40
    while i > 0:
        index = randint(0, len(opt_list)-1)
        if count_list[index] > 0:
            count_list[index] = count_list[index] - 1
            order_list.append(opt_list[index])
            i -= 1
        else:
            continue
    return order_list

In [45]:
for fldr in os.listdir(order_dirs):
    for i in range(1, 101):
        opt_list = get_order(opt_dict[fldr])
        txt_path = os.path.join(order_dirs, fldr, f'order_{i:03}.txt')
        with open(txt_path, 'w') as file:
            file.writelines(f'{opt}/n' for opt in opt_list)

# Other Testing

In [7]:
one_opt_list = os.listdir('/Users/AP-CNL/Desktop/4RT/Stimuli/Order/1_option')
random.shuffle(one_opt_list)
one_opt_list[0]

'order_036.txt'

In [17]:
txt_path = '/Users/AP-CNL/Desktop/4RT/Stimuli/Order/4_options/order_001.txt'
with open(txt_path, 'r') as file:
    order_list = [line.strip() for line in file]

order_list = [item for pair in zip(order_list, ['feedback']*len(order_list)) for item in pair][:-1]
print(order_list)

['circle', 'feedback', 'line', 'feedback', 'triangle', 'feedback', 'square', 'feedback', 'line', 'feedback', 'circle', 'feedback', 'circle', 'feedback', 'square', 'feedback', 'circle', 'feedback', 'line', 'feedback', 'square', 'feedback', 'circle', 'feedback', 'square', 'feedback', 'square', 'feedback', 'line', 'feedback', 'triangle', 'feedback', 'triangle', 'feedback', 'line', 'feedback', 'square', 'feedback', 'triangle', 'feedback', 'line', 'feedback', 'triangle', 'feedback', 'triangle', 'feedback', 'triangle', 'feedback', 'triangle', 'feedback', 'line', 'feedback', 'line', 'feedback', 'square', 'feedback', 'square', 'feedback', 'circle', 'feedback', 'square', 'feedback', 'square', 'feedback', 'triangle', 'feedback', 'line', 'feedback', 'triangle', 'feedback', 'line', 'feedback', 'circle', 'feedback', 'circle', 'feedback', 'circle', 'feedback', 'circle']
